In [2]:
import torch
import numpy as np
import scipy
import matplotlib.pyplot as plt
import os

gpu = torch.device("cuda:0")
print(torch.cuda.get_device_name(0))

# Use smaller network for testing - ex 2000 neurons
# Even for the project, doing it for 10^6 neurons would take too long
# Problem this creates: test network is denser than actual network b/c we have 10^3 neurons but 10^2 connections per neuron
num_neurons = 2000
num_i = int(0.1 * num_neurons)
num_e = int(0.9 * num_neurons)

# Epsilon value close to 0 to prevent nan in division by 0
eps = 1e-6

# Num excitatory inputs and inhibitory inputs to each neuron (in reality it should be 500 but we reduce it here to make things faster)
k = 100

# Number of olfactory bulb channels (glomeruli) to each neuron
D = 10 ** 3
# For each neuron, how many glomeruli inputs it receives (should be 10^2)
num_channel_inputs = 100

# Number of odors
P = 16
# Novel activity is up to P // 2, and familiar activity is after
novel_inds = torch.arange(0, P // 2)
familiar_inds = torch.arange(P // 2, P)

# %%
# Creates sparse adjacency matrix with the given probability of edge connection and size mxn
def create_adj_matrix(p, m, n):
    # num_connections = int(p * m * n)
    # m_coords = torch.randint(0, m, (num_connections,))
    # n_coords = torch.randint(0, n, (num_connections,))
    # indices = torch.vstack((m_coords, n_coords))
    # values = torch.ones(num_connections)
    # A_mn = torch.sparse_coo_tensor(indices, values, (m, n))
    probs = torch.ones(m, n) * p
    A_mn = torch.bernoulli(probs)
    return A_mn

# New way of generating correlations between odors: we want different sets of odors to be correlated differently, so that when we subtract each neuron's mean activity over odors, it doesn't cancel out the variation between odors (if all the odors are correlated the same, they will tend to produce similar values for a single neuron and therefore subtracting by the mean will remove these values and only leave small fluctuations)
# So we sample a small set of odors P' and make them linearly independent, and then by multiplying by a P'x P gaussian matrix we project into mitral cell activity space for all P odors, basically making the P odors a linear combination of the set of P' odors (the smaller P' is, the more correlated the resulting set of P odors will be)
# We also scale the variance depending on how small P' is, so we will maintain differently correlated odors, just with higher total correlation if P' is small
P_prime = 4
def correlated_mitral_activity():
    # Each of the P' odors is independent (correlation of 0)
    sigma_p_prime = torch.zeros((P_prime, P_prime)).fill_diagonal_(1)
    dist = torch.distributions.MultivariateNormal(torch.zeros(P_prime), sigma_p_prime)
    p_prime_activity = dist.sample(torch.Size([D]))
    var = 1 / P_prime
    projection = torch.normal(torch.zeros((P_prime, P)), torch.ones(P_prime, P) * np.sqrt(var))
    activity = p_prime_activity @ projection
    return activity.to(gpu)

# Takes in mitral activity I and feedforward weights W_ff and computes feedforward activity h_bar_ff
def compute_feedforward_activity(W_ff, I):
    with torch.device(gpu):
        h_ff = (W_ff @ I) * (1 / np.sqrt(num_channel_inputs))
        h_bar_ff = torch.zeros_like(h_ff)
        # Subtract by mean across (excitatory) neurons for each odor
        h_bar_ff[:num_e] = h_ff[:num_e] - torch.mean(h_ff[:num_e], dim=0, keepdim=True)
    return h_bar_ff

# Computes feedforward (channel) weights mapping mitral activity onto E,I neurons
def compute_feedforward_weights():
    # Probability that a channel weight will be nonzero
    p = num_channel_inputs / D
    with torch.device(gpu):
        a = create_adj_matrix(p, num_e, D)
        # Inhibitory neurons don't receive channel input
        # This is the first simplification, where we neglect the first inhibitory layer I_ff
        b = torch.zeros(size=(num_i, D))
        W_ff = torch.cat(tensors=(a, b), dim=0)

    return W_ff

def compute_initial_recurrent_weights():
    k_ee = k_ei = k_ie = k_ii = k
    #p_ee = k_ee / num_e
    # k inhibitory inputs to that e neuron, out of num_i total inhibitory neurons gives the connection probability per neuron
    p_ei = k_ei / num_i
    p_ie = k_ie / num_e
    #p_ii = k_ii / num_i
    
    # Constants
    #w_ee = 0.1
    w_ei = 0.2
    w_ie = 0.5
    #w_ii = 0.3
    # Ignore ee and ii weights for now:
    p_ee = p_ii = w_ee = w_ii = 0
    with torch.device(gpu):
        W_ee = create_adj_matrix(p_ee, num_e, num_e) * w_ee
        W_ei = create_adj_matrix(p_ei, num_e, num_i) * -w_ei
        W_ie = create_adj_matrix(p_ie, num_i, num_e) * w_ie
        W_ii = create_adj_matrix(p_ii, num_i, num_i) * -w_ii
        
        # Concat
        W_1 = torch.cat(tensors=(W_ee, W_ei), dim=1)
        W_2 = torch.cat(tensors=(W_ie, W_ii), dim=1)
        W_rec = torch.cat(tensors=(W_1, W_2), dim=0)
    
    return W_rec

# Computes activation threshold for neurons, right now set it at 0
def compute_threshold():
    threshold = torch.zeros((num_neurons, P), device=gpu)
    # Since inhibitory neurons are linear
    threshold[num_e:, :] = 0
    return threshold

# ReLU for excitatory, linear for inhibitory
def neuron_activations(X):
    # Mask to keep excitatory
    mask1 = torch.ones((num_neurons, 1), device=gpu)
    mask1[num_e:, :] = 0
    # Mask to keep inhibitory
    mask2 = torch.zeros((num_neurons, 1), device=gpu)
    mask2[num_e:, :] = 1
    return (torch.relu(X) * mask1) + (X * mask2)

# %%
# Computes R for each odor, with the activation threshold theta
def compute_piriform_response(h_bar_ff, W_rec):
    # The coefficient of x_bar
    tau = 1
    # time step
    dt = 0.1
    # Number of time steps
    T = 200
    
    # Initial condition where states are gaussian
    mu_0 = 0.
    sigma_0 = 0.2
    X_0 = torch.normal(mu_0, sigma_0, size=(num_neurons, P))
    X = X_0.to(gpu)
    
    pts = []
    for i in range(T-2):
        with torch.no_grad():
            part1 = -1 * X
            part2 = (W_rec @ neuron_activations(X)) * (1 / np.sqrt(k))
            part3 = h_bar_ff
            dXdt = (1 / tau) * (part1 + part2 + part3)
            X = X + (dXdt * dt)
        # Look at convergence pattern for first odor, assuming that it'll
        # be similar across odors (since they are all independent)
        #pts.append(torch.mean(dXdt, dim=0)[0].item())
   
    # On the last 2 iterations only, track the gradient
    X.requires_grad_(True)
    
    for j in range(2):
        part1 = -1 * X
        part2 = (W_rec @ neuron_activations(X)) * (1 / np.sqrt(k))
        part3 = h_bar_ff
        dXdt = (1 / tau) * (part1 + part2 + part3)
        X = X + (dXdt * dt)
    
    # The total input to the neuron at this last time step (should be equivalent to the resulting value of X after this time step, since dxdt = 0 after the recurrent network converges)
    #total_input = part2 + part3
    threshold = compute_threshold()
    
    # Plot derivatives to see if state converged
    # plt.plot(torch.arange(T-2), pts)
    # plt.show()
    R = neuron_activations(X - threshold)
    
    return R


# Start and stop indices for the section of W_rec we want to update, respectively 
# Takes in R matrix (neuron responses for each odor and tuple of update inds representing ie, then ei (each element in that tuple is itself a tuple of (post, pre))
def compute_updates(R: torch.Tensor, models: tuple, update_inds: tuple) -> torch.Tensor:
    postsyn_responses = R[update_inds[i][0], :]
    presyn_responses = R[update_inds[i][1], :]
    
    model_input = torch.stack(tensors=(presyn_responses, postsyn_responses), dim=2).transpose(1, 0)
    #updates_per_odor = models[i](model_input)

    model_updates = torch.mean(updates_per_odor, dim=0).squeeze(dim=1)
    
    return all_updates

NVIDIA GeForce RTX 3060


In [3]:
ie_post = (num_e, num_neurons)
ie_pre = (0, num_e)
def get_update_inds(post, pre, W):
    weights_slice = W[post[0]:post[1], pre[0]:pre[1]]
    inds = torch.nonzero(weights_slice, as_tuple=True)
    update_inds = (inds[0] + post[0], inds[1] + pre[0])
    
    return update_inds

In [5]:
W_ff = compute_feedforward_weights()
I = correlated_mitral_activity()
hbar_ff = compute_feedforward_activity(W_ff, I)
W_rec = compute_initial_recurrent_weights()
R = compute_piriform_response(hbar_ff, W_rec)
update_inds = get_update_inds(ie_post, ie_pre, W_rec)

In [69]:
def powerseries(W, R, update_inds):
    # First do for the single-neuron case:
    # Degree of polynomial
    degree = 2
    # Num vars in polynomial (presyn, postsyn, weight)
    numvars = 3
    # Num terms in polynomial is numvars ^ (degree + 1)
    numterms = numvars ** (degree + 1)
    # Confavreux paper: N(0, 0.1) = N(0, 0.3162^2)
    mu = torch.ones(numterms)
    std = torch.ones(numterms) * 0.3162
    A = torch.normal(mu, std).to(gpu)

    post = R[update_inds[0], :]
    pre = R[update_inds[1], :]
    ws = torch.repeat_interleave(W[update_inds[0], update_inds[1]].unsqueeze(1), repeats=P, dim=1)
    post = post.expand(numterms, -1, -1)
    pre = pre.expand(numterms, -1, -1)
    #print(post.shape)
    #ws = torch.repeat_interleave(ws.unsqueeze(0), repeats=numterms, dim=0)
    ws = ws.expand(numterms, -1, -1)
    combined = torch.stack((post, pre, ws), dim=1)
    exps = torch.cartesian_prod(*([torch.arange(degree + 1)] * numvars)).to(gpu)
    exps = exps.unsqueeze(2).unsqueeze(3)
    exps = exps.broadcast_to(combined.shape)
    result = torch.pow(combined, exps)
    sum_terms = torch.prod(result, dim=1)
    A_coef = A.unsqueeze(1).unsqueeze(2)
    result = A_coef * sum_terms
    updates_per_odor = torch.sum(result, dim=0)
    updates = torch.mean(updates_per_odor, dim=1)

    return updates

powerseries(W_rec, R, update_inds)

torch.Size([20135])


In [70]:
# [0, 0, 0] [1, 1, 1] [2, 2, 2]
# [0, 1, 2] [0, 1, 2] [0, 1, 2]
# [0, 1, 2] [0, 1, 2] [0, 1, 2]

# degree = 2
# numvars = 3
# exps = torch.cartesian_prod(*([torch.arange(degree + 1)] * numvars))
# print(exps)

